# Chapter 2 — Attention to GPT
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch02_transformer_gpt.ipynb)

Implements scaled dot-product attention, causal masking, multi-head attention, LayerNorm/GELU feed-forward blocks, and a small GPT. Designed for a Colab T4.

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.cuda.get_device_name(0) if device=='cuda' else '')

## 1. Scaled dot-product attention

In [ ]:
def attention(q, k, v, causal=True):
    scores = q @ k.transpose(-2,-1) / math.sqrt(q.size(-1))
    if causal:
        T = q.size(-2)
        mask = torch.triu(torch.ones(T,T,device=q.device,dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
    w = F.softmax(scores, dim=-1)
    return w @ v, w

x = torch.randn(2,8,32,device=device)
y,w = attention(x,x,x)
print(y.shape, w.shape)

## 2. Multi-head causal self-attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=128, n_head=4):
        super().__init__(); assert d_model % n_head == 0
        self.n_head=n_head; self.head_dim=d_model//n_head
        self.qkv=nn.Linear(d_model,3*d_model); self.proj=nn.Linear(d_model,d_model)
    def forward(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,dim=-1)
        def split(z): return z.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

m=CausalSelfAttention().to(device)
print(m(torch.randn(2,64,128,device=device)).shape)

## 3. GPT block and model

In [ ]:
class Block(nn.Module):
    def __init__(self,d_model=128,n_head=4,mlp_ratio=4):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model); self.attn=CausalSelfAttention(d_model,n_head)
        self.ln2=nn.LayerNorm(d_model)
        self.mlp=nn.Sequential(nn.Linear(d_model,mlp_ratio*d_model),nn.GELU(),nn.Linear(mlp_ratio*d_model,d_model))
    def forward(self,x):
        x=x+self.attn(self.ln1(x)); x=x+self.mlp(self.ln2(x)); return x

class TinyGPT(nn.Module):
    def __init__(self,vocab=1000,ctx=256,d_model=128,n_head=4,n_layer=4):
        super().__init__(); self.ctx=ctx
        self.tok=nn.Embedding(vocab,d_model); self.pos=nn.Embedding(ctx,d_model)
        self.blocks=nn.ModuleList([Block(d_model,n_head) for _ in range(n_layer)])
        self.ln=nn.LayerNorm(d_model); self.head=nn.Linear(d_model,vocab,bias=False)
        self.head.weight=self.tok.weight
    def forward(self,idx,targets=None):
        B,T=idx.shape; pos=torch.arange(T,device=idx.device)
        x=self.tok(idx)+self.pos(pos)
        for b in self.blocks: x=b(x)
        logits=self.head(self.ln(x))
        loss=None if targets is None else F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1))
        return logits,loss

model=TinyGPT().to(device)
idx=torch.randint(0,1000,(4,128),device=device)
logits,loss=model(idx,idx)
loss.backward()
print(logits.shape, float(loss), 'params=',sum(p.numel() for p in model.parameters()))

## T4 note
The book's chapter-2 scale is also small enough for T4. This notebook uses an even smaller model so forward/backward completes quickly.

Upstream: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch02